# Laboratorio Evaluado: WBCD y Evaluación Multimodelo
**Alumno:** Alacoria Cueto Emanuel

**Reto Autónomo: Clínica Oncológica**

Eres el Consultor IA de un hospital. El dataset WBCD (Wisconsin Breast Cancer Diagnostic) contiene radios y perímetros de células.
Se entrenan k-NN y Árboles de Decisión, y se valida qué pasa si no se escalan los datos.

**Dataset:** [Breast Cancer Wisconsin (Diagnostic)](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic)


## 1. Carga de Datos y Exploración

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Nombres de columnas del dataset WBCD (wdbc.data no trae encabezado)
column_names = ['id', 'diagnosis']
for suffix in ['mean', 'se', 'worst']:
    for feature in ['radius', 'texture', 'perimeter', 'area', 'smoothness',
                    'compactness', 'concavity', 'concave_points', 'symmetry',
                    'fractal_dimension']:
        column_names.append(f'{feature}_{suffix}')

# Carga del dataset desde UCI
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data'
df_wbcd = pd.read_csv(url, header=None, names=column_names)

print('Dimensiones del dataset:', df_wbcd.shape)
df_wbcd.head()

In [ ]:
print('Tipos de datos:')
print(df_wbcd.dtypes.value_counts())
print()
print('Valores nulos por columna:')
print(df_wbcd.isnull().sum().sum(), 'nulos en total')
print()
print('Distribución del target (diagnosis):')
print(df_wbcd['diagnosis'].value_counts())
print(df_wbcd['diagnosis'].value_counts(normalize=True).round(3))
print()
print('Estadísticas descriptivas de algunas variables (radio y perímetro):')
df_wbcd[['radius_mean', 'radius_worst', 'perimeter_mean', 'perimeter_worst']].describe().round(2)

## 2. Preprocesamiento

Se convierte la columna `diagnosis` a binario:
- **M** (Maligno) → **1** (caso positivo, la clase que queremos detectar)
- **B** (Benigno) → **0** (caso negativo)

La columna `id` se descarta porque no aporta información predictiva.

La división es **80% train / 20% test** con `random_state=42` para reproducibilidad y `stratify=y` para mantener la proporción de clases en ambos conjuntos.

In [ ]:
# Conversión de diagnosis (M/B) a binario (1/0)
df_wbcd['diagnosis'] = df_wbcd['diagnosis'].map({'M': 1, 'B': 0})

# Separación en X (features) e y (target)
X = df_wbcd.drop(columns=['id', 'diagnosis'])
y = df_wbcd['diagnosis']

# Train-test split: 80% entrenamiento, 20% prueba
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print('Tamaño train:', X_train.shape)
print('Tamaño test :', X_test.shape)
print('Proporción de malignos en train:', round(y_train.mean(), 3))
print('Proporción de malignos en test :', round(y_test.mean(), 3))

## 3. Escalamiento (Solo para k-NN)

k-NN es un algoritmo **basado en distancias**: clasifica un punto según los vecinos más cercanos en el espacio de características. Si las variables tienen escalas muy diferentes (por ejemplo, `area_mean` oscila en cientos y `smoothness_mean` en décimas), las variables con magnitudes grandes **dominarán la distancia euclidiana** y las demás quedarán invisibles para el modelo.

Se usa `StandardScaler`, que transforma cada variable a media 0 y desviación estándar 1.

⚠️ **Regla de oro:** el scaler se ajusta **solo con los datos de entrenamiento** (`.fit_transform()`), y al conjunto de prueba solo se le aplica la transformación (`.transform()`). Esto evita **data leakage**: información del test nunca debe influir en el preprocesamiento.

Nota: el Árbol de Decisión **no requiere escalamiento**, porque sus splits se basan en umbrales de una sola variable a la vez, no en distancias.

In [ ]:
# Escalamiento con StandardScaler
scaler = StandardScaler()

# Fit solo con train, transform en ambos
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print('Media de X_train_scaled (aprox. 0):', X_train_scaled.mean().round(6))
print('Desviación estándar de X_train_scaled (aprox. 1):', X_train_scaled.std().round(6))

## 4. Modelado y Matrices de Confusión

Se entrenan dos modelos:

| Modelo | Configuración | ¿Necesita escalamiento? |
|---|---|---|
| k-NN | `KNeighborsClassifier(n_neighbors=5)` | ✅ Sí (distancias) |
| Árbol de Decisión | `DecisionTreeClassifier(max_depth=4)` | ❌ No (umbrales por variable) |

La **matriz de confusión** se organiza así:

```
                Predicho 0 (B)   Predicho 1 (M)
Real 0 (B)      TN               FP
Real 1 (M)      FN  ← Error Tipo II   TP
```

- **TN** (True Negative): benigno correctamente clasificado.
- **FP** (False Positive / Error Tipo I): benigno clasificado como maligno → genera ansiedad y estudios adicionales, pero no cuesta la vida.
- **FN** (False Negative / Error Tipo II): **maligno clasificado como benigno** → el paciente se va a casa sin tratamiento; es el error más grave.
- **TP** (True Positive): maligno correctamente detectado.

In [ ]:
# --- Modelo 1: k-NN (con datos escalados) ---
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)
y_pred_knn = knn.predict(X_test_scaled)

# --- Modelo 2: Árbol de Decisión (con datos sin escalar) ---
tree = DecisionTreeClassifier(max_depth=4, random_state=42)
tree.fit(X_train, y_train)
y_pred_tree = tree.predict(X_test)

# --- Matrices de confusión ---
cm_knn = confusion_matrix(y_test, y_pred_knn)
cm_tree = confusion_matrix(y_test, y_pred_tree)

print('Matriz de confusión k-NN (escalado):')
print(cm_knn)
print()
print('Matriz de confusión Árbol de Decisión (sin escalar):')
print(cm_tree)

In [ ]:
print('=== k-NN (escalado) ===')
print(classification_report(y_test, y_pred_knn, target_names=['Benigno (0)', 'Maligno (1)']))
print('Accuracy k-NN:', round(accuracy_score(y_test, y_pred_knn), 4))
print()
print('=== Árbol de Decisión (sin escalar) ===')
print(classification_report(y_test, y_pred_tree, target_names=['Benigno (0)', 'Maligno (1)']))
print('Accuracy Árbol:', round(accuracy_score(y_test, y_pred_tree), 4))

In [ ]:
# Resumen cuantitativo de errores clínicos (Falsos Negativos y Falsos Positivos)
def errores_clinicos(cm, nombre):
    tn, fp, fn, tp = cm.ravel()
    print(f'{nombre}:')
    print(f'  Falsos Negativos (malignos no detectados): {fn}')
    print(f'  Falsos Positivos (benignos alarmados):     {fp}')
    print(f'  Sensibilidad (recall maligno):             {tp/(tp+fn):.3f}')
    print()

errores_clinicos(cm_knn, 'k-NN (escalado)')
errores_clinicos(cm_tree, 'Árbol de Decisión (sin escalar)')

## 5. Validación: ¿Qué pasa si NO escalas los datos?

Para demostrar la importancia del escalamiento se entrena un **k-NN adicional con los datos crudos (sin escalar)** y se compara con el k-NN escalado. La diferencia en accuracy y en la matriz de confusión muestra el efecto directo de las magnitudes de las variables.

In [ ]:
# k-NN SIN escalamiento (datos crudos)
knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_raw.fit(X_train, y_train)
y_pred_knn_raw = knn_raw.predict(X_test)

cm_knn_raw = confusion_matrix(y_test, y_pred_knn_raw)

print('Matriz de confusión k-NN SIN escalar:')
print(cm_knn_raw)
print()
print('Accuracy k-NN escalado :', round(accuracy_score(y_test, y_pred_knn), 4))
print('Accuracy k-NN sin escalar:', round(accuracy_score(y_test, y_pred_knn_raw), 4))
print()
errores_clinicos(cm_knn_raw, 'k-NN SIN escalar')

# Visualización comparativa
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, cm, title in zip(axes,
                         [cm_knn_raw, cm_knn, cm_tree],
                         ['k-NN sin escalar', 'k-NN escalado', 'Árbol (sin escalar)']):
    im = ax.imshow(cm, cmap='Blues')
    for (i, j), val in np.ndenumerate(cm):
        ax.text(j, i, val, ha='center', va='center', fontsize=14,
                color='white' if val > cm.max()/2 else 'black')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Pred B', 'Pred M']); ax.set_yticklabels(['Real B', 'Real M'])
    ax.set_title(title)
plt.tight_layout()
plt.show()

## Evaluación Teórica / Decisión de Negocios

**1. En este escenario médico, ¿qué implica un Error Tipo II (Falso Negativo) y por qué cuesta vidas?**

Un Error Tipo II (Falso Negativo) ocurre cuando el modelo clasifica como **benigno** un tumor que en realidad es **maligno**. En términos clínicos, esto significa que a un paciente con cáncer de mama se le dice que está sano, por lo que **no recibe tratamiento** (cirugía, quimioterapia ni radioterapia) ni seguimiento oportuno. El cáncer continúa progresando silenciosamente y, cuando finalmente se detecta, la enfermedad puede estar en etapas avanzadas donde la supervivencia es mucho menor. Por eso un Falso Negativo cuesta vidas: directamente retrasa el diagnóstico y el tratamiento de una enfermedad que es curable en fases tempranas (la tasa de supervivencia a 5 años del cáncer de mama localizado supera el 90%, pero cae drásticamente si se disemina). En este laboratorio el FN aparece en la celda [1,0] de la matriz de confusión (real maligno, predicho benigno). Por ello, en un sistema médico real se prioriza la **sensibilidad (recall)** sobre la precisión o la accuracy global, aunque eso signifique tolerar más Falsos Positivos.

**2. ¿Por qué crees que el Árbol de Decisión no falló estrepitosamente incluso si no hubieses escalado los datos?**

El Árbol de Decisión **no se basa en distancias**, sino en **comparaciones de umbral sobre una sola variable a la vez**: para decidir un split pregunta, por ejemplo, "¿radius_mean ≤ 17.3?". Este tipo de comparación es **invariante a transformaciones monótonas** de las variables: si `radius_mean` está en milímetros o si está estandarizado (z-scores), el punto de corte óptimo cambia de valor numérico pero la **estructura del árbol y su capacidad de separar clases permanecen idénticas**. En otras palabras, el escalamiento no altera el orden de los datos ni la pureza de los grupos que el árbol puede formar. Además, el árbol selecciona solo las variables con mayor poder discriminante (como `concave points_worst` o `radius_worst`) e ignora el resto, por lo que las magnitudes de las demás variables son irrelevantes. En contraste, k-NN sí colapsa sin escalamiento porque las variables de gran magnitud (área, perímetro) dominan la distancia euclidiana y las variables de pequeña magnitud (suavidad, simetría) dejan de influir en la clasificación.

**Conclusión para la clínica:** para este problema médico, k-NN escalado suele ofrecer una excelente sensibilidad, pero el Árbol de Decisión tiene la ventaja adicional de ser **interpretable** (se puede mostrar al oncólogo la regla de decisión), lo cual es valioso en un contexto clínico regulado. Independientemente del modelo elegido, la métrica de negocio clave debe ser el número de **Falsos Negativos**, no la accuracy global.